# MapLibreum: GeoJSON and Choropleth Maps

This notebook shows how to load and display external GeoJSON datasets, apply dynamic styling, add tooltips based on feature properties, and create choropleth maps.

## 1. Loading GeoJSON Data

MapLibreum makes it easy to add spatial vector data in the GeoJSON format. You can pass a dictionary, a file path, or an HTTP URL.

In [ ]:
import json
from maplibreum import Map, GeoJson

# Let's load the outlines of all US states from a local file
url = 'data/us-states.json'

with open(url) as response:
    states_data = json.load(response)

m1 = Map(center=[-96, 37.8], zoom=4)

# Add the GeoJSON to the map with default styling
GeoJson(states_data, name="US States").add_to(m1)

m1

## 2. Dynamic Styling with Callbacks

Instead of a uniform style, you can use a Python function to compute the style for each feature dynamically based on its properties.

In [ ]:
m2 = Map(center=[-96, 37.8], zoom=4)

# Let's define a style function that colors states differently if their name starts with 'M'
def style_function(feature):
    state_name = feature['properties'].get('name', '')
    
    if state_name.startswith('M'):
        return {
            'fillColor': '#ff7800',
            'color': '#000000',
            'weight': 2,
            'fillOpacity': 0.6
        }
    else:
        return {
            'fillColor': '#3388ff',
            'color': '#000000',
            'weight': 1,
            'fillOpacity': 0.2
        }

GeoJson(
    states_data,
    style_function=style_function,
    name="Styled States"
).add_to(m2)

m2

## 3. Adding Interactivity (Tooltips & Popups)

You can attach `GeoJsonTooltip` or `GeoJsonPopup` to display properties when a user hovers over or clicks a feature.

In [ ]:
from maplibreum import GeoJsonTooltip, GeoJsonPopup

m3 = Map(center=[-96, 37.8], zoom=4)

# Configure a tooltip to show specific fields
tooltip = GeoJsonTooltip(
    fields=['name', 'density'],
    aliases=['State:', 'Population Density:'],
    labels=True
)

# Configure a popup to show when clicked
popup = GeoJsonPopup(
    fields=['name'],
    aliases=['State']
)

GeoJson(
    states_data,
    tooltip=tooltip,
    popup=popup,
    style_function=lambda x: {'fillColor': 'green', 'weight': 1, 'fillOpacity': 0.4}
).add_to(m3)

m3

## 4. Choropleth Maps

A Choropleth map binds data values (e.g. population density) to geographic regions by shading them with a color scale.

In [ ]:
import pandas as pd
from maplibreum.choropleth import Choropleth
from maplibreum.core import LayerControl

# We load some sample state employment data to go with our states GeoJSON
unemployment_url = 'data/US_Unemployment_Oct2012.csv'
state_data = pd.read_csv(unemployment_url)

m4 = Map(center=[-102, 48], zoom=3)

# Create the choropleth layer
# Note: Convert state_data to dictionary format expected by maplibreum.Choropleth
data_dict = state_data.set_index('State')['Unemployment'].to_dict()

Choropleth(
    geojson=states_data,
    data=data_dict,
    key_on='name', # Use 'name' property from GeoJSON to match with the dictionary keys
    color_scale='linear',
    legend_title='Unemployment Rate (%)'
).add_to(m4)

LayerControl().add_to(m4)

m4